# FloraSense C3: root and leaf distribution — Colab GPU

Upload `florasense_c3_training.zip` to **My Drive / FloraSense_C3**. Open this
notebook with Colab, then select **Runtime → Change runtime type → T4 GPU**.
Run the cells in order. Training runs only in hosted Colab with CUDA; do not use
a local runtime. The ZIP contains original verified image bytes, relative-path
manifests, audit provenance, and a snapshot of the training implementation.

The baseline trains two independent two-class ResNet18 heads on frozen ImageNet
features. Three outer grouped folds are evaluated once; separate inner validation
groups select checkpoints and early stopping. No test-fold model selection.
P08 and stem height are outside this experiment. No existing model is loaded.


In [ ]:
import os, platform, sys
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
assert platform.system() != "Windows", "Use a hosted Google Colab runtime"
from google.colab import drive
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
drive.mount("/content/drive")


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib, json, shutil, stat, zipfile

DRIVE_DIR = Path("/content/drive/MyDrive/FloraSense_C3")
DATA_ZIP = DRIVE_DIR / "florasense_c3_training.zip"
EXPECTED_ZIP_SHA256 = "4cae788b6d48b52d6203d85b46ed59328de7544e997d72376124d927a8f95015"
assert DATA_ZIP.is_file(), f"Upload the prepared ZIP to {DATA_ZIP}"
LOCAL_ZIP = Path("/content/florasense_c3_training.zip")
shutil.copyfile(DATA_ZIP, LOCAL_ZIP)
def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
assert sha256(LOCAL_ZIP) == EXPECTED_ZIP_SHA256, "Wrong or incomplete ZIP; use the matching notebook"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DATA_ROOT = Path("/content") / ("florasense_c3_" + stamp)
DATA_ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)), "Duplicate archive members"
    for member in archive.infolist():
        name = PurePosixPath(member.filename)
        assert not name.is_absolute() and ".." not in name.parts
        assert ":" not in member.filename and "\\" not in member.filename
        assert not stat.S_ISLNK(member.external_attr >> 16), "Symlinks are not accepted"
    archive.extractall(DATA_ROOT)
bundle = json.loads((DATA_ROOT / "bundle.json").read_text())
assert set(names) == set(bundle["file_sha256"]) | {"bundle.json"}
for name, expected in bundle["file_sha256"].items():
    assert sha256(DATA_ROOT / name) == expected, f"Checksum mismatch: {name}"
print("Verified", bundle["image_files"], "image files; source workbook:", bundle["workbook_sha256"])


In [ ]:
import subprocess
# Colab supplies the CUDA-compatible torch/torchvision pair. Do not replace it
# with local CPU wheels. Install only the data/evaluation dependencies.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(DATA_ROOT / "requirements_colab.txt")])
sys.path.insert(0, str(DATA_ROOT))


In [ ]:
import numpy as np
import pandas as pd
from services.c3_quality_assessment.train_feature_classifier import (
    CLASSES, load_verified_manifest, grouped_folds, main,
)
from services.c3_quality_assessment.prepare_feature_data import validate_split_groups

for task in ("root", "leaf"):
    frame = load_verified_manifest(DATA_ROOT / f"{task}_usable.csv", task)
    labels = frame.label.map({name: i for i, name in enumerate(CLASSES[task])}).to_numpy()
    print(task, "samples:", len(frame), "labels:", frame.label.value_counts().to_dict())
    for fold, (development, test) in enumerate(grouped_folds(frame, labels, 3, 42, minimum=3), 1):
        train_rel, val_rel = grouped_folds(frame.iloc[development].reset_index(drop=True), labels[development], 3, 42 + fold, minimum=2)[0]
        train, validation = development[train_rel], development[val_rel]
        for a, b in ((train, validation), (train, test), (validation, test)):
            validate_split_groups(frame.iloc[a], frame.iloc[b])
        print(" fold", fold, {name: np.bincount(labels[idx], minlength=2).tolist()
                              for name, idx in (("train", train), ("validation", validation), ("test", test))})
print("Preflight passed. No model has been trained by these setup cells.")


## Train on the GPU

The next cell starts training. Defaults: seed 42, 3 outer folds, 3-way inner
split (one held-out validation fold), up to 40 epochs, patience 6, AdamW 0.001,
weight decay 0.01, batch size 16. Only the final linear layer is trained; the
ImageNet backbone and BatchNorm stay frozen in evaluation mode. Two fixed
augmentation views plus the original are used for training only. Frozen encoder
features are cached in memory; no encoder is fitted on validation/test data.

Images are EXIF-oriented, resized to fit 208×208, and centered on a 224×224 gray
canvas. Training uses horizontal flips, ±4° rotation, and mild color changes;
no crops or vertical flips. Class weights use training groups only. Argmax is
fixed; no test-based threshold tuning. All views/crops share `sample_key`.

Checkpoints are saved to Drive after each fold. The final model is refit on all
usable data for the median validation-selected epoch count. CV estimates the
training procedure; the final refit has no independent external test score.


In [ ]:
RUN_DIR = DRIVE_DIR / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
print("Saving models and reports to", RUN_DIR)
main(["--task", "both", "--manifest-dir", str(DATA_ROOT), "--output-dir", str(RUN_DIR),
      "--folds", "3", "--seed", "42", "--max-epochs", "40", "--patience", "6",
      "--learning-rate", "0.001", "--augmentation-views", "2"])


In [ ]:
import matplotlib.pyplot as plt
summary_rows = []
for task in ("root", "leaf"):
    report = json.loads((RUN_DIR / task / "evaluation.json").read_text())
    metrics = report["oof_metrics"]
    summary_rows.append({"task": task, **{k: metrics[k] for k in
        ("macro_f1", "balanced_accuracy", "minority_precision", "minority_recall")}})
    print(task, "per-class precision/recall/F1/support:")
    display(pd.DataFrame(metrics["classification_report"]).T)
    print("Outer-fold variation (each test fold evaluated once):")
    display(pd.DataFrame([{"fold": item["fold"], "best_epoch": item["best_epoch"],
        **{key: item["metrics"][key] for key in ("macro_f1", "balanced_accuracy", "minority_precision", "minority_recall")}}
        for item in report["folds"]]))
    print("Per plant (both-class metrics unavailable where a class is absent):")
    display(pd.DataFrame([{"plant": plant, "samples": value["samples"],
        "support": value["class_support"], "macro_f1": value["metrics"]["macro_f1"],
        "balanced_accuracy": value["metrics"]["balanced_accuracy"]}
        for plant, value in report["per_plant"].items()]))
    matrix = np.asarray(metrics["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(matrix, cmap="Blues")
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=CLASSES[task], yticklabels=CLASSES[task],
           xlabel="Predicted", ylabel="True", title=f"{task}: out-of-fold confusion matrix")
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, str(value), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(RUN_DIR / task / "confusion_matrix.png", dpi=160)
    plt.show()
display(pd.DataFrame(summary_rows))


In [ ]:
from google.colab import files
artifact_zip = shutil.make_archive("/content/florasense_c3_models", "zip", root_dir=RUN_DIR)
drive_copy = RUN_DIR.parent / (RUN_DIR.name + "_models.zip")
shutil.copyfile(artifact_zip, drive_copy)
print("Drive backup:", drive_copy)
print("Final checkpoints:", RUN_DIR / "root/final_model.pth", RUN_DIR / "leaf/final_model.pth")
files.download(artifact_zip)


## Interpretation and next handoff

Download includes separate root/leaf `final_model.pth`, three validation-selected
fold checkpoints per task, split CSVs, out-of-fold predictions, evaluation JSON,
class mappings, preprocessing metadata, and confusion-matrix PNGs. Keep the
metadata and reports with the weights. No general-quality checkpoint is present.

Minority counts are only 15 root and 16 leaf samples. Report fold variability and
majority baselines, not just pooled accuracy. There is no P08 evidence and no
P05 One Side examples. Composite keys assume each record identifies one physical
plant; exact hashes prevent byte-identical leakage but cannot establish that
different photos show different plants. Review cross-ID recaptures/near-duplicates
before making generalization claims. Do not tune on outer test predictions.

If Colab disconnects, saved Drive fold checkpoints remain. Restart with a new
output directory; this initial pipeline reruns rather than resuming optimizer
state. GPU availability, including T4 availability, depends on Colab allocation.
API/frontend integration and stem-height assessment are separate future phases.
